In [3]:
import torch

data = torch.load("outputs/sae/analysis_z_con.pt", weights_only=False)

In [4]:
print(data.keys())
# dict_keys(['stats', 'correlations', 'exemplars', 'cooccurrence',
#  'top_cooccurring_pairs', 'decoder_similarity', 'decoder_clusters',
#  'probe_alignment', 'element_enrichment', 'dashboard', 'n_dead',
#  'n_rare', 'H_val', 'history'])

# Accessing each part

# 1. Activation stats — all ndarray (n_features,)
data["stats"]["fire_rate"]       # fraction of crystals activating each feature
data["stats"]["mean_act"]        # mean activation when active
data["stats"]["std_act"]         # std when active
data["stats"]["max_act"]         # max activation


dict_keys(['stats', 'correlations', 'exemplars', 'cooccurrence', 'top_cooccurring_pairs', 'decoder_similarity', 'decoder_clusters', 'probe_alignment', 'element_enrichment', 'dashboard', 'n_dead', 'n_rare', 'sim_threshold', 'H_val', 'history'])


array([0.54880106, 0.56553364, 1.4472474 , ..., 0.59355515, 0.72972584,
       0.6450829 ], dtype=float32)

In [5]:
data["correlations"]["band_gap"]          # Pearson r for each feature vs band gap
data["correlations"]["formation_energy"]  # etc.

# 3. Top exemplars — dict[int, list[int]]
data["exemplars"][347]   # crystal indices that most activate feature 347

# 4. Co-occurrence — ndarray (n_features, n_features)
data["cooccurrence"][12, 347]  # fraction of crystals where both 12 and 347 fire

# Top pairs as a list of (feat_i, feat_j, rate)
data["top_cooccurring_pairs"][:5]

# 5. Decoder similarity — ndarray (n_features, n_features)
data["decoder_similarity"][12, 347]  # cosine sim between feature directions

# Clusters of similar features
data["decoder_clusters"]  # list of lists, e.g. [[12, 347, 901], [55, 2003]]

# 6. Probe alignment (if probe weights were provided)
data["probe_alignment"]["band_gap"]  # cosine sim between each SAE feature and probe direction

# 7. Element enrichment (if dataset was provided)
data["element_enrichment"][347]  # [("O", 3.2), ("Ti", 2.8), ...] — enrichment ratios

# 8. Dashboard — list of dicts, top 30 features by correlation
for feat in data["dashboard"][:5]:
    print(f"Feature {feat['feature_idx']}: "
        f"{feat['top_property']} r={feat['top_corr']:.3f}, "
        f"fires {feat['fire_rate']*100:.1f}%")

# 9. Sparse activations for all val crystals — Tensor (N_val, n_features)
data["H_val"].shape  # e.g. (9047, 4096)

# 10. Training history
data["history"]["train_loss"]     # list of floats, one per epoch
data["history"]["var_explained"]  # list of floats, one per epoch

Feature 3289: formation_energy r=-0.647, fires 18.4%
Feature 2363: formation_energy r=-0.616, fires 16.4%
Feature 3003: formation_energy r=-0.604, fires 21.5%
Feature 3913: formation_energy r=-0.585, fires 19.1%
Feature 416: formation_energy r=-0.569, fires 14.5%


[-1.6046507735935553,
 -1.3750999964127195,
 -1.1761424461828507,
 -1.0051325228971932,
 -0.8568619291371862,
 -0.7274980856861077,
 -0.6134351170588515,
 -0.5126590325515101,
 -0.4237449922377603,
 -0.3436654243415811,
 -0.27173138505520344,
 -0.20713600744046912,
 -0.14795657112697858,
 -0.09382202889302649,
 -0.04427765613134804,
 0.0018827411750769185,
 0.04474028360280424,
 0.08491356397248473,
 0.12250469968612454,
 0.15812833213227873,
 0.19250579136608992,
 0.22503607231368672,
 0.2560037555650304,
 0.28608554323629887,
 0.31477541958204236,
 0.34170869066353304,
 0.3677429722036897,
 0.39234055952123514,
 0.41586258251810515,
 0.43833643004225786,
 0.45974211249603214,
 0.4800222953745078,
 0.4990828803584615,
 0.5178521440741728,
 0.535187464073928,
 0.5518707977778696,
 0.5677178612401159,
 0.582842974814734,
 0.5973061460342382,
 0.6113749488347653,
 0.6242853234591952,
 0.6368492504636928,
 0.6489195196329228,
 0.6603533757975404,
 0.671520432816628,
 0.682061846420541,
 0

In [6]:
import numpy as np

corrs = data["correlations"]["band_gap"]
top_10 = np.argsort(np.abs(corrs))[::-1][:10]

for idx in top_10:
    print(f"  Feature {idx:4d}  r={corrs[idx]:+.4f}  "
        f"fire_rate={data['stats']['fire_rate'][idx]*100:.1f}%")

# %%
# ---- Step 1: Pick a feature to investigate ----
# Start with the dashboard (top 30 by correlation)
for feat in data["dashboard"]:
    elems = ""
    if "enriched_elements" in feat and feat["enriched_elements"]:
        elems = ", ".join(f"{s}({r:.1f}x)" for s, r in feat["enriched_elements"][:3])
    print(
        f"Feature {feat['feature_idx']:4d}  "
        f"fire={feat['fire_rate']*100:5.1f}%  "
        f"top={feat['top_property']:>20s} r={feat['top_corr']:+.3f}  "
        f"elems=[{elems}]"
    )

  Feature 2171  r=+0.4389  fire_rate=24.5%
  Feature 1494  r=+0.4356  fire_rate=8.6%
  Feature 3140  r=+0.4340  fire_rate=22.2%
  Feature  416  r=+0.4197  fire_rate=14.5%
  Feature 2363  r=+0.4045  fire_rate=16.4%
  Feature 3003  r=+0.4023  fire_rate=21.5%
  Feature 1629  r=+0.3917  fire_rate=13.8%
  Feature 3913  r=+0.3374  fire_rate=19.1%
  Feature 2522  r=+0.3331  fire_rate=13.1%
  Feature 3289  r=+0.3284  fire_rate=18.4%
Feature 3289  fire= 18.4%  top=    formation_energy r=-0.647  elems=[Dy(7.2x), Lu(5.9x), Nd(5.4x)]
Feature 2363  fire= 16.4%  top=    formation_energy r=-0.616  elems=[Ti(10.5x), Si(6.1x), Mg(6.0x)]
Feature 3003  fire= 21.5%  top=    formation_energy r=-0.604  elems=[F(14.1x), Mn(9.9x), K(8.2x)]
Feature 3913  fire= 19.1%  top=    formation_energy r=-0.585  elems=[Bi(9.9x), Pa(8.6x), Hf(7.3x)]
Feature  416  fire= 14.5%  top=    formation_energy r=-0.569  elems=[Pa(8.6x), Sc(7.1x), Hf(6.1x)]
Feature 1494  fire=  8.6%  top=    formation_energy r=-0.562  elems=[Pa(21.5

In [7]:
from pathlib import Path
import csv

def inspect_feature(feat_idx, data, csv_path="data/mp_20/val.csv", out_file="feature_report.txt"):
    stats = data["stats"]
    corrs = data["correlations"]

    def log(s=""):
        print(s)
        with open(out_file, "a") as f:
            f.write(s + "\n")

    log(f"\n{'='*60}")
    log(f"FEATURE {feat_idx}")
    log(f"{'='*60}")
    log(f"Fire rate:  {stats['fire_rate'][feat_idx]*100:.1f}%")
    log(f"Mean act:   {stats['mean_act'][feat_idx]:.4f}")
    log(f"Std act:    {stats['std_act'][feat_idx]:.4f}")
    log(f"Max act:    {stats['max_act'][feat_idx]:.4f}")
    log(f"\nProperty correlations:")
    for prop in sorted(corrs.keys()):
        r = corrs[prop][feat_idx]
        bar = "█" * int(abs(r) * 30)
        log(f"  {prop:>25s}  r={r:+.4f}  {bar}")
    if data["probe_alignment"] is not None:
        log(f"\nProbe direction alignment:")
        for prop, vals in data["probe_alignment"].items():
            log(f"  {prop:>25s}  cos={vals[feat_idx]:+.4f}")
    if data["element_enrichment"] is not None and feat_idx in data["element_enrichment"]:
        elems = data["element_enrichment"][feat_idx]
        if elems:
            log(f"\nEnriched elements (vs dataset baseline):")
            for sym, ratio in elems:
                log(f"  {sym:>3s}  {ratio:.2f}x")
    cooc = data["cooccurrence"]
    cooc_row = cooc[feat_idx].copy()
    cooc_row[feat_idx] = 0
    top_cooc = np.argsort(cooc_row)[::-1][:5]
    log(f"\nTop co-occurring features:")
    for j in top_cooc:
        log(f"  Feature {j:4d}  co-occur={cooc_row[j]*100:.1f}%")
    exemplars = data["exemplars"].get(feat_idx, [])
    if exemplars and Path(csv_path).exists():
        rows = list(csv.DictReader(open(csv_path)))
        log(f"\nTop activating crystals:")
        log(f"  {'Index':>6s}  {'Formula':>15s}  {'SG':>4s}  {'Ef':>8s}  {'Eg':>6s}  {'Ehull':>6s}")
        for ci in exemplars[:10]:
            if ci < len(rows):
                r = rows[ci]
                log(
                    f"  {ci:6d}  {r['pretty_formula']:>15s}  "
                    f"{r['spacegroup.number']:>4s}  "
                    f"{float(r['formation_energy_per_atom']):>8.3f}  "
                    f"{float(r['band_gap']):>6.3f}  "
                    f"{float(r['e_above_hull']):>6.3f}"
                )



In [8]:
for i in range(len(data["dashboard"])):
    inspect_feature(data["dashboard"][i]["feature_idx"], data)


FEATURE 3289
Fire rate:  18.4%
Mean act:   1.3740
Std act:    0.7581
Max act:    4.3970

Property correlations:
                   band_gap  r=+0.3284  █████████
             crystal_system  r=-0.2031  ██████
               e_above_hull  r=+0.1166  ███
           formation_energy  r=-0.6471  ███████████████████
                   is_metal  r=-0.2837  ████████
                  num_atoms  r=+0.3373  ██████████
                 spacegroup  r=-0.2055  ██████

Probe direction alignment:
           formation_energy  cos=-0.1586
                   band_gap  cos=+0.1032
               e_above_hull  cos=+0.0091
                  num_atoms  cos=+0.1622

Enriched elements (vs dataset baseline):
   Dy  7.18x
   Lu  5.93x
   Nd  5.40x
    Y  4.86x
   La  4.72x

Top co-occurring features:
  Feature  965  co-occur=12.5%
  Feature 3003  co-occur=10.8%
  Feature 2363  co-occur=9.6%
  Feature 3913  co-occur=9.3%
  Feature 1452  co-occur=9.3%

FEATURE 2363
Fire rate:  16.4%
Mean act:   1.2590
Std act: 